> # SOLUTIONS — Block 5 — LLM Validation
>
> **Facilitator copy.** This notebook contains the answers to every fill-in and discussion prompt in the participant version. Hold it back until after the workshop. Use during the session for spot-checks and for unblocking participants who get stuck.
>
> The participant notebook is `block_5_—_llm_validation.ipynb`. Differences from this version are limited to (a) blanks filled in, (b) discussion answers added inline, and (c) this banner.

---


# Block 5 — Validating the Model with LLM-Assisted Spec Review

**ODSC Tutorial: Spec-Driven Simulation Modeling**

A simulation that runs is not the same as a simulation that is right. The expensive bugs in simulation work do not crash; they produce plausible-looking wrong answers. This block uses an LLM to read the generated code against the specification and surface deviations.

We are going to feed the LLM three things:

1. **Template A** — the spec, written before any code existed.
2. **The SimPy code** that was generated from it.
3. **The baseline results** the code produced.

We will ask the LLM to flag any deviation between spec and code, plus any reasonableness concerns about the results. Then we triage what came back.

The supplied code contains two intentionally seeded issues, planted so we can see what LLM-assisted review can and cannot find. One of them is invisible in the summary metrics — exactly the bug class this technique is best at catching.

## Setup

Standard imports plus a small helper that calls an LLM if you have a key configured, and otherwise falls back to a saved representative response. The fallback exists so this exercise still works on conference Wi-Fi or without an API key.

If you want to run the live API call, set `ANTHROPIC_API_KEY` in your environment before launching Jupyter. If you don't, the cell will use the saved response without changing the participant experience.

In [1]:
import json
import os
import textwrap
from pathlib import Path

# Paths to the three artifacts the LLM will review.
ROOT = Path("..").resolve()
TEMPLATE_A_PATH = ROOT / "templates" / "coffee_shop_des_template_a_completed.md"
CODE_PATH       = ROOT / "code"      / "coffee_shop_des_buggy.py"
RESULTS_PATH    = ROOT / "code"      / "baseline_results_summary.json"
SAVED_RESPONSE  = Path("llm_responses") / "spec_compliance_review.json"

## 1. Load the three review inputs

If any of these files is missing on your machine, the cell will tell you so you can fix the path and re-run. The most common cause is running the notebook from a directory other than `notebooks/`.

In [2]:
def _read_text(p: Path) -> str:
    if not p.exists():
        raise FileNotFoundError(f"Expected to find {p}. Adjust the path constants in the cell above and re-run.")
    return p.read_text()

template_a_md = _read_text(TEMPLATE_A_PATH)
buggy_code    = _read_text(CODE_PATH)

# baseline_results_summary.json is small; if it's not present we synthesise
# a representative summary so the exercise still works.
if RESULTS_PATH.exists():
    baseline_results = json.loads(_read_text(RESULTS_PATH))
else:
    baseline_results = {
        "scenario": "Baseline",
        "n_replications": 50,
        "mean_wait_min": 1.89,
        "p90_wait_min": 5.98,
        "walkaway_rate_pct": 1.21,
        "mean_served_per_day": 148.1,
        "mean_walkaways_per_day": 1.95,
        "max_queue_length": 8,
        "morning_utilization": 0.835,
    }

print(f"Template A:        {len(template_a_md):,} chars")
print(f"SimPy code:        {len(buggy_code):,} chars")
print(f"Baseline results:  {list(baseline_results.keys())}")

Template A:        15,748 chars
SimPy code:        26,729 chars
Baseline results:  ['scenario', 'n_replications', 'mean_wait_min', 'p90_wait_min', 'walkaway_rate_pct', 'mean_served_per_day', 'mean_walkaways_per_day', 'morning_utilization', 'max_queue_length_rep0', 'daily_cost_usd']


## 2. Build the spec-compliance review prompt

The prompt is the work product. It tells the LLM exactly what to do, what evidence to ground each claim in, and what shape of response to return. A vague "review this code" gets you a generic code review. A prompt that names the artifacts, the review criteria, and a response schema gets you something you can triage.

Three design choices worth pointing out:

- **Ground every issue in the spec.** Each finding has to cite the section of Template A it deviates from. This keeps the LLM honest and makes false positives easier to spot.
- **Ask for severity *and* category.** Severity sorts the triage queue. Category (spec_deviation, missing_validation, false_positive_risk) tells you what kind of follow-up the issue needs.
- **Demand a structured response.** JSON with named fields beats free-form prose. It is much easier to triage 4 issues in a table than to scan 600 words of analysis.

In [3]:
PROMPT_TEMPLATE = '''
You are reviewing a discrete event simulation against its specification. Your job is to flag any deviation between the spec and the code, plus any reasonableness concerns about the produced results.

# SPECIFICATION (Template A)

{template_a}

# IMPLEMENTATION

```python
{code}
```

# BASELINE RESULTS

{results}

# YOUR TASK

Identify issues in two categories:

1. **Spec deviations.** Anywhere the code does not faithfully implement what the spec describes. For each, cite the section of the spec, quote the relevant code, and explain why the deviation matters even if the summary metrics look plausible.

2. **Missing input validation or constraint checks.** Places where the spec implies a constraint (positive durations, non-negative counts, monotonic ordering) that the code does not enforce.

For each issue, return: `id`, `severity` (high|medium|low), `category` (spec_deviation|missing_validation|reasonableness), `title`, `evidence`, `why_it_matters`, and `fix`.

Also return a short `summary` and a list of `false_positives_to_watch_for` -- issues another reviewer might raise that are out-of-scope per the spec.

Respond with a single JSON object. Do not include any prose outside the JSON.
'''.strip()

prompt = PROMPT_TEMPLATE.format(
    template_a=template_a_md,
    code=buggy_code,
    results=json.dumps(baseline_results, indent=2),
)
print(f"Prompt length: {len(prompt):,} chars")

Prompt length: 43,937 chars


## 3. Send the prompt to an LLM (with saved-response fallback)

The helper below tries to call Anthropic's API if a key is set in the environment, parses the JSON response, and returns it. If anything fails — no API key, network error, parse error — it transparently falls back to a saved representative response so the exercise can continue.

The saved response was captured from a real LLM run on the exact same prompt. It is not synthetic. The fallback is honest about which path was taken.

In [4]:
def call_llm_live(prompt: str) -> dict:
    """Call Anthropic's API. Raises if no key set or call fails."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError("ANTHROPIC_API_KEY not set")
    # Lazy-import so the notebook still loads if the package isn't installed.
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    msg = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}],
    )
    text = msg.content[0].text
    # Strip a possible code fence in case the model added one.
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text)


def call_llm_with_fallback(prompt: str) -> tuple[dict, str]:
    """Try live; fall back to the saved response. Returns (response, source)."""
    try:
        return call_llm_live(prompt), "live_api"
    except Exception as e:
        print(f"[fallback] live call unavailable ({type(e).__name__}: {e}); using saved response")
        return json.loads(SAVED_RESPONSE.read_text()), "saved_fallback"


review, source = call_llm_with_fallback(prompt)
print(f"Source: {source}")
print(f"Issues flagged: {len(review.get('issues', []))}")
print(f"Observations:   {len(review.get('observations_not_bugs', []))}")

[fallback] live call unavailable (RuntimeError: ANTHROPIC_API_KEY not set); using saved response
Source: saved_fallback
Issues flagged: 2
Observations:   3


## 4. Read what came back

Render the structured response as a triage table. For each flagged issue, look at:

- **Evidence.** Does the LLM ground its claim in something specific in the spec or code? Or is it gesturing at a generic concern?
- **Why it matters.** Does the explanation hold up if you read it carefully? Or does it sound plausible but fall apart under scrutiny?
- **Fix.** Is the proposed fix concrete and small? Or is it a vague rewrite?

Then decide for each issue: **real bug**, **valid observation but not a bug**, or **false positive**.

In [5]:
def render_issue(issue: dict, idx: int) -> None:
    print("=" * 72)
    print(f"[{idx}] {issue.get('id','?')}  {issue.get('severity','?').upper()}  ({issue.get('category','?')})")
    print(f"     {issue.get('title','(no title)')}")
    print("-" * 72)
    print("Evidence:")
    print(textwrap.fill(issue.get('evidence',''), width=72, initial_indent="  ", subsequent_indent="  "))
    print()
    print("Why it matters:")
    print(textwrap.fill(issue.get('why_it_matters',''), width=72, initial_indent="  ", subsequent_indent="  "))
    print()
    print("Suggested fix:")
    print(textwrap.fill(issue.get('fix',''), width=72, initial_indent="  ", subsequent_indent="  "))
    print()


print(f"SUMMARY: {review.get('summary','(no summary)')}\n")

print("=" * 72)
print("ISSUES")
print("=" * 72)
for i, issue in enumerate(review.get('issues', []), start=1):
    render_issue(issue, i)

if review.get('observations_not_bugs'):
    print("=" * 72)
    print("OBSERVATIONS (not bugs, but worth knowing)")
    print("=" * 72)
    for o in review['observations_not_bugs']:
        print(f"  [{o.get('id','?')}] {o.get('note','')}")
        print()

if review.get('false_positives_to_watch_for'):
    print("=" * 72)
    print("FALSE POSITIVES THE MODEL ITSELF FLAGS")
    print("=" * 72)
    for fp in review['false_positives_to_watch_for']:
        print(f"  - {fp}")

SUMMARY: Reviewed coffee_shop_des_buggy.py against Template A (the coffee shop process and resource spec) and the supplied baseline results. Found two issues that warrant attention: one spec deviation in how balking is implemented, and one missing input-validation check. Also noted three observations about validation hygiene that are not bugs but are worth flagging for completeness.

ISSUES
[1] B1  HIGH  (spec_deviation)
     Balking check fires after yield req, not at arrival
------------------------------------------------------------------------
Evidence:
  Template A, Section 5 (Queue Behavior): 'Balking — a customer who
  arrives and sees the line at length ≥ 8 turns around and leaves
  immediately. The balking decision is made at arrival, before the
  customer joins the line.' The implementation in entity_process places
  the balking check inside the `with baristas.request() as req:` block,
  after `yield req` has already returned. By that point the customer has
  waited in the q

## 5. Triage — your turn

For each flagged issue, decide where it goes:

- **Real bug** — the code violates the spec and the deviation matters. Fix it.
- **Valid observation, not a bug** — the spec doesn't require what the LLM is asking for, but the comment is useful. File it.
- **False positive** — the LLM is asking for something the spec explicitly excludes, or is hallucinating a constraint that doesn't exist. Discard it.

Fill in the dictionary below with your call on each issue, then re-run to see the triage table.

In [6]:
# Replace each "?" with one of: "real_bug", "valid_observation", "false_positive"
participant_triage = {
    "B1": "real_bug",   # balking is checked AFTER yield req — spec says "at arrival"
    "B2": "real_bug",   # spec asserts service_time > 0; code does not enforce it
}

def show_triage(triage: dict, issues: list[dict]) -> None:
    print(f"{'ID':<5}{'Title':<55}{'Your call':<20}")
    print("-" * 80)
    for issue in issues:
        idv = issue.get('id', '?')
        title = issue.get('title', '')[:53]
        call = triage.get(idv, "?")
        print(f"{idv:<5}{title:<55}{call:<20}")

show_triage(participant_triage, review.get('issues', []))

ID   Title                                                  Your call           
--------------------------------------------------------------------------------
B1   Balking check fires after yield req, not at arrival    real_bug            
B2   No guard that drawn service times are positive         real_bug            


> ### Facilitator notes — triage answers
>
> **B1 (balking placement) — real_bug.** The spec says balking is at arrival; the code checks after `yield req`. The walk-away semantics are wrong even though the metrics look plausible. This is the headline teaching example of the bug class spec-driven validation catches and runtime testing tends to miss.
>
> **B2 (positivity check) — real_bug.** The spec calls out a service-time positivity constraint that the code does not enforce. Lognormal samples are mathematically positive, so this defect is dormant under the current configuration — but the moment someone changes the distribution to one that allows zero or negative draws, the bug activates silently. A defensive `assert` is a one-line fix.
>
> **The three "observations" in the saved response — valid_observation, not bugs.** Warm-up of 30 minutes is a deliberate non-stationary-system choice; the M/M/c sanity check passes; replication count is reasonable. None violate the spec.
>
> **The "false_positives_to_watch_for" entry (reneging) — definitively false_positive.** Template A explicitly excludes reneging. Any LLM that flags the lack of reneging handling is misreading the scope, not finding a bug. This is a useful teaching moment about LLM limitations: high-confidence findings still need triage.
>
> **The point of the exercise.** Participants should leave with: *the LLM finds candidates; you decide what's a bug.* Not "the LLM is right" and not "LLMs hallucinate so don't use them." The skill is in the triage step.


## 6. Discussion

Two questions to take to the room:

**The walkaway-rate signal is faint.** Walkaway rate moved from about 0.9% under the spec-compliant version to about 1.3% under the buggy version. Maximum queue length is identical (8 in both). P90 wait shifted by under a minute. None of these would jump out in a normal "did the simulation produce sane numbers" review. If you only had the metrics dashboard, would you have caught this bug?

**What kind of bug *is* this, exactly?** The simulation runs without error. Customers do walk away. The rate is in the right order of magnitude. The defect is that walkaways are happening *at the wrong moment in the customer journey* — after waiting through the queue rather than at arrival. That is a behavioral correctness issue with downstream consequences (customer experience, queue dynamics under load, and what the simulation says happens when you change the balking threshold) but it is invisible at the metrics layer. This is precisely the bug class spec-driven validation with LLM review exists to catch.

In Block 6 we use the validated simulation to compare the four staffing scenarios and answer the actual decision question.